In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

# =========================
# 1. BACA DATA
# =========================
df = pd.read_csv("/kaggle/input/datasets/yooojii/data-analisis-regresi/financials.csv")

# Pilih peubah
x_col = "52 Week Low"
y_col = "Price"

# Ambil 30 amatan acak
sample = df.sample(n=30, random_state=42)[["Symbol", "Name", y_col, x_col]].reset_index(drop=True)

print("=== 30 Amatan yang Digunakan ===")
print(sample.to_string(index=True))

# =========================
# 2. HITUNG KOMPONEN MANUAL
# =========================
x = sample[x_col].astype(float).values
y = sample[y_col].astype(float).values

n = len(sample)

sum_x = np.sum(x)
sum_y = np.sum(y)
sum_x2 = np.sum(x**2)
sum_y2 = np.sum(y**2)
sum_xy = np.sum(x*y)

x_bar = np.mean(x)
y_bar = np.mean(y)

Sxx = sum_x2 - (sum_x**2 / n)
Syy = sum_y2 - (sum_y**2 / n)
Sxy = sum_xy - (sum_x * sum_y / n)

b1 = Sxy / Sxx
b0 = y_bar - b1 * x_bar

SSR = b1 * Sxy
SSE = Syy - SSR
SST = Syy

MSR = SSR / 1
MSE = SSE / (n - 2)

F_hit = MSR / MSE
R2 = SSR / SST
R2_adj = 1 - (SSE / (n - 2)) / (SST / (n - 1))

# Standard error
se_b1 = np.sqrt(MSE / Sxx)
se_b0 = np.sqrt(MSE * (1/n + (x_bar**2 / Sxx)))

# t hitung
t_b1 = b1 / se_b1
t_b0 = b0 / se_b0

# p-value
p_b1 = 2 * (1 - stats.t.cdf(abs(t_b1), df=n-2))
p_b0 = 2 * (1 - stats.t.cdf(abs(t_b0), df=n-2))

alpha = 0.05
t_tabel = stats.t.ppf(1 - alpha/2, df=n-2)
F_tabel = stats.f.ppf(1 - alpha, dfn=1, dfd=n-2)

# Confidence interval beta
ci_b1 = (b1 - t_tabel * se_b1, b1 + t_tabel * se_b1)
ci_b0 = (b0 - t_tabel * se_b0, b0 + t_tabel * se_b0)

# =========================
# 3. PREDIKSI PADA x0
# =========================
x0 = 100
yhat_x0 = b0 + b1 * x0

se_mean = np.sqrt(MSE * (1/n + ((x0 - x_bar)**2 / Sxx)))
se_pred = np.sqrt(MSE * (1 + 1/n + ((x0 - x_bar)**2 / Sxx)))

ci_mean = (yhat_x0 - t_tabel * se_mean, yhat_x0 + t_tabel * se_mean)
ci_individu = (yhat_x0 - t_tabel * se_pred, yhat_x0 + t_tabel * se_pred)

# =========================
# 4. TAMPILKAN HASIL MANUAL
# =========================
print("\n=== RINGKASAN PERHITUNGAN MANUAL ===")
print(f"n = {n}")
print(f"sum_x = {sum_x:.6f}")
print(f"sum_y = {sum_y:.6f}")
print(f"sum_x2 = {sum_x2:.6f}")
print(f"sum_y2 = {sum_y2:.6f}")
print(f"sum_xy = {sum_xy:.6f}")
print(f"x_bar = {x_bar:.6f}")
print(f"y_bar = {y_bar:.6f}")
print(f"Sxx = {Sxx:.6f}")
print(f"Syy = {Syy:.6f}")
print(f"Sxy = {Sxy:.6f}")

print("\n=== PARAMETER REGRESI ===")
print(f"b0 (intercept) = {b0:.6f}")
print(f"b1 (slope)     = {b1:.6f}")
print(f"Persamaan regresi: Y_hat = {b0:.6f} + {b1:.6f}X")

print("\n=== ANOVA ===")
print(f"SSR = {SSR:.6f}")
print(f"SSE = {SSE:.6f}")
print(f"SST = {SST:.6f}")
print(f"MSR = {MSR:.6f}")
print(f"MSE = {MSE:.6f}")
print(f"F_hit = {F_hit:.6f}")
print(f"F_tabel = {F_tabel:.6f}")

print("\n=== UJI HIPOTESIS PARAMETER ===")
print(f"se(b1) = {se_b1:.6f}")
print(f"t_hit beta1 = {t_b1:.6f}")
print(f"p-value beta1 = {p_b1:.10f}")

print(f"se(b0) = {se_b0:.6f}")
print(f"t_hit beta0 = {t_b0:.6f}")
print(f"p-value beta0 = {p_b0:.10f}")

print(f"t_tabel = {t_tabel:.6f}")

print("\n=== SELANG KEPERCAYAAN 95% ===")
print(f"CI beta1 = ({ci_b1[0]:.6f}, {ci_b1[1]:.6f})")
print(f"CI beta0 = ({ci_b0[0]:.6f}, {ci_b0[1]:.6f})")

print("\n=== KOEFISIEN DETERMINASI ===")
print(f"R^2 = {R2:.6f}")
print(f"Adjusted R^2 = {R2_adj:.6f}")

print("\n=== PREDIKSI UNTUK x0 = 100 ===")
print(f"Y_hat(100) = {yhat_x0:.6f}")
print(f"CI mean response = ({ci_mean[0]:.6f}, {ci_mean[1]:.6f})")
print(f"CI individual response = ({ci_individu[0]:.6f}, {ci_individu[1]:.6f})")

# =========================
# 5. REGRESI DENGAN STATSMODELS
# =========================
X = sm.add_constant(sample[x_col])
model = sm.OLS(sample[y_col], X).fit()

print("\n=== OUTPUT STATSMODELS ===")
print(model.summary())

=== 30 Amatan yang Digunakan ===
   Symbol                            Name    Price  52 Week Low
0     EFX                    Equifax Inc.   114.00      147.020
1      KR                      Kroger Co.    27.57       34.750
2      WY              Weyerhaeuser Corp.    33.60       37.890
3    BIIB                     Biogen Inc.   311.79      370.570
4     MET                    MetLife Inc.    44.28       56.580
5     BWA                      BorgWarner    51.94       58.220
6    VIAB                     Viacom Inc.    32.71       46.720
7     DVA                     DaVita Inc.    71.91       80.710
8     VAR          Varian Medical Systems   112.82      130.290
9      XL                      XL Capital    41.26       47.270
10    GWW            Grainger (W.W.) Inc.   258.60      298.145
11    AMD      Advanced Micro Devices Inc    11.22       15.650
12    WMT                 Wal-Mart Stores   100.02      109.980
13    BSX               Boston Scientific    25.20       29.930
14   ND